In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your OS:
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. In a terminal, start a local model:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To test that the local server is running, try:
   ```bash
   curl http://localhost:11434
   ```

If you get a connection refused error while prompting Ollama in the homework, restart the Ollama server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [6]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

If you want to run the course locally, you can do so if you’re comfortable setting up the needed tools yourself.

For local setup, make sure you have:
- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

Also, document your setup and keep the environment reproducible.

If you meant **Olama** specifically, I don’t have any FAQ entry for that name in the provided context.


In [7]:
## Before we see how LLM can use search, lets see the below example. Here we just send a query to OpenAI

In [8]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment policy and whether registration is still open.\n\nIf you want, I can help you figure it out quickly. Tell me:\n- the course name\n- the school/platform\n- whether it’s live, self-paced, or in-person\n\nIn general, you may be able to join if:\n- enrollment is still open\n- there are no prerequisites you haven’t met\n- the course hasn’t already ended\n- there’s still space, if it has a cap\n\nIf you’re asking for a message you can send, here’s a simple one:\n\n> Hi, I just discovered this course and I’m interested in joining. Is it still possible to enroll? If so, could you please share the next steps?\n\nIf you want, I can help you write a more specific email or message.'

In [9]:
messages = [
    {'role': 'user', 'content': 'How do I run Ollama locally?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'To run Ollama locally:\n\n### 1) Install Ollama\n- **macOS / Windows:** download from https://ollama.com\n- **Linux:** use the install command from the Ollama site, or:\n```bash\ncurl -fsSL https://ollama.com/install.sh | sh\n```\n\n### 2) Start the Ollama service\nUsually it starts automatically after install. If needed, run:\n```bash\nollama serve\n```\n\n### 3) Pull a model\nFor example:\n```bash\nollama pull llama3.1\n```\n\n### 4) Run the model\n```bash\nollama run llama3.1\n```\n\nThis opens an interactive chat in your terminal.\n\n### 5) Use it via API\nOllama runs a local HTTP server at `http://localhost:11434`. Example:\n```bash\ncurl http://localhost:11434/api/generate -d \'{\n  "model": "llama3.1",\n  "prompt": "Write a haiku about cats"\n}\'\n```\n\n### Useful commands\n```bash\nollama list       # installed models\nollama ps         # running models\nollama stop llama3.1\n```\n\nIf you want, I can also show you how to run Ollama in Docker or connect it from Python.'

In [10]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [11]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [12]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [13]:
len(response.output)

1

In [14]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"How do I run Ollama locally?"}', call_id='call_nwzeNEAcnlMfuQcMIE0AiFSV', name='search', type='function_call', id='fc_07cd9761ad67ec92006a84519148ec87d2896dbf9dc2b17a05', namespace=None, status='completed')]

In [15]:
call = response.output[0]

In [16]:
call

ResponseFunctionToolCall(arguments='{"query":"How do I run Ollama locally?"}', call_id='call_nwzeNEAcnlMfuQcMIE0AiFSV', name='search', type='function_call', id='fc_07cd9761ad67ec92006a84519148ec87d2896dbf9dc2b17a05', namespace=None, status='completed')

In [ ]:
## The response contains a function_call entry. 
## The model decided it needs to search the FAQ before answering. 
## Rather than reply, it asked us to run the search function first.
## The function call contains JSON arguments. We parse them, call our search function, and serialize the result.

In [17]:
import json

args = json.loads(call.arguments)
args


{'query': 'How do I run Ollama locally?'}

In [18]:
args

{'query': 'How do I run Ollama locally?'}

In [19]:
call.name

'search'

In [20]:
results = search(**args)

In [21]:
results

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [30]:
result_json = json.dumps(results, indent=2)

In [ ]:
## Now we send this result back to the model

In [39]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [40]:
messages.append(call)

In [41]:
messages.append(function_call_output)

In [42]:
messages

[{'role': 'user', 'content': 'How do I run Ollama locally?'},
 ResponseFunctionToolCall(arguments='{"query":"How do I run Ollama locally?"}', call_id='call_nwzeNEAcnlMfuQcMIE0AiFSV', name='search', type='function_call', id='fc_07cd9761ad67ec92006a84519148ec87d2896dbf9dc2b17a05', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"How do I run Ollama locally?"}', call_id='call_nwzeNEAcnlMfuQcMIE0AiFSV', name='search', type='function_call', id='fc_07cd9761ad67ec92006a84519148ec87d2896dbf9dc2b17a05', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_nwzeNEAcnlMfuQcMIE0AiFSV',
  'output': '[\n  {\n    "id": "1d0b969028",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Ollama: How to install Ollama?",\n    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\\n\\n- **macOS**: Download the `.pkg` and i

In [43]:
## Put your original question in its own cell/variable, separate from the loop, so you can always re-run the reset line without 
## retyping the question

question = "I just discovered the course. Can I join now?"
messages = [{"role": "user", "content": question}]

In [44]:
## If you want to double check state before resetting, you can inspect what's currently in there
print(len(messages))
for m in messages:
    print(m)

1
{'role': 'user', 'content': 'I just discovered the course. Can I join now?'}


In [45]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [46]:
messages.append(call)

In [47]:
messages.append(function_call_output)

In [48]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join now?'},
 ResponseFunctionToolCall(arguments='{"query":"How do I run Ollama locally?"}', call_id='call_nwzeNEAcnlMfuQcMIE0AiFSV', name='search', type='function_call', id='fc_07cd9761ad67ec92006a84519148ec87d2896dbf9dc2b17a05', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_nwzeNEAcnlMfuQcMIE0AiFSV',
  'output': '[\n  {\n    "id": "1d0b969028",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Ollama: How to install Ollama?",\n    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\\n\\n- **macOS**: Download the `.pkg` and install it.\\n- **Windows**: Download the `.msi` and install it.\\n- **Linux**: Run the following command in the terminal:\\n\\n  ```bash\\n  curl -fsSL https://ollama.com/install.sh | sh\\n  ```\\n\\nOnce installed, open a terminal and ty

In [49]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [50]:
print(response.output_text)

Yes — you can join now.

If you mean the course itself, it’s okay to start when you discover it; you don’t need to wait for a special date to begin learning. You can also run it locally if you prefer, instead of using Codespaces.

If you want, I can help you with the very first step, like setting up the environment.


# Token usage and cost

In [ ]:
## We just made two API calls instead of one. 
## Each call we send to the model costs money, so it's worth checking how much one tool-using turn actually costs.
## The response has a usage field with the token counts

In [52]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(1378, 78)

In [54]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


In [55]:
# Your tokens
result = calculate_gpt54mini_price(1378, 78)

In [56]:
print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0002535


In [ ]:
# This usage is only for the second API call. The first call also has its own usage and its own cost. That was the call where the model decided to invoke search. 
# Two calls means we pay twice. We pay even more on the second call, because we resend the full history as input.

# A developer prompt

In [ ]:
# So far we've relied on the model to figure out when to search. 
# We make that more reliable with a developer message that spells out how to behave. 
# This is where we give the agent its role. 
# The same message also pushes it toward multiple searches, so we get to watch the loop run more than once.

In [69]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [75]:
call

ResponseFunctionToolCall(arguments='{"query":"How do I run Ollama locally?"}', call_id='call_nwzeNEAcnlMfuQcMIE0AiFSV', name='search', type='function_call', id='fc_07cd9761ad67ec92006a84519148ec87d2896dbf9dc2b17a05', namespace=None, status='completed')

# A function-call helper

In [71]:
# We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. 
# It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. 
# We only have one tool for now, so we dispatch on the function name directly.

In [72]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

# Processing one response

In [73]:
# Let's process a single model response. We append each output entry to the conversation, print any messages, and run any function calls. 
# Function-call results get appended too.

In [77]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late enrollment discovered the course can I join"}
function_call: search {"query":"course enrollment can I join after course starts discovered the course"}
function_call: search {"query":"FAQ join the course after it has started enrollment access"}


In [76]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course enroll late join student FAQ"}', call_id='call_8e4uXcjo48ZfPdnvTYN8Jqnh', name='search', type='function_call', id='fc_051a72e7c01ee5a0006a846cc7bf34819ea91ff6a80ffd690c', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_8e4uXcjo48ZfPdnvTYN8Jqnh',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "c

# The full agent loop


In [ ]:
# he loop keeps calling the model until it returns a response without any function calls. 
# We also keep an iteration counter so we can see how many round-trips happened.

In [78]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while the course is still accepting submissions. You can also start learning now and work through the materials at your own pace.

If you want, I can also help with:
- how to start the course,
- whether you can still get a certificate,
- or how homework/project submissions work.

Is there another area you want to explore?


In [ ]:
# This is the core agent loop. 
# The model reasons about the next action. Your code performs it, and the model sees the result on the next turn. 
# The loop stops when the model returns a final answer with no more tool calls.
# We don't decide how many times the model searches. The model does, and we keep looping until it stops asking for tools.

# The exit condition is the simplest one possible. No function calls this turn means we're done. 
# Other frameworks add safety nets on top, like a max iteration count, a token budget, or a wall-clock limit. 
# You might cap it at five iterations and force an answer on the last one. The core is still this one flag.

# Wrapping it in a function

In [ ]:
# Let's wrap the loop in a function so we can reuse it. 
# The function takes the instructions and the question as parameters, and returns the final answer.

In [79]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [80]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install Ollama local FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: download and install the `.pkg`
     - **Windows**: download and install the `.msi`
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   - In a terminal, run:
     ```bash
     ollama run llama3
     ```
   - This will download the model and start a local chat interface.

3. **Check that the local server is running**
   - You can test it with:
     ```bash
     curl http://localhost:11434
     ```
   - You should get a response showing available models.

4. **Use it from Python**
   - Install the client:
     ```bash
     pip install ollama
     ```
   - Example:
     ```python
     import ollama

     response = ollama.chat(
         model='llama3',
         messa

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: download and install the `.pkg`\n     - **Windows**: download and install the `.msi`\n     - **Linux**: run:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model locally**\n   - In a terminal, run:\n     ```bash\n     ollama run llama3\n     ```\n   - This will download the model and start a local chat interface.\n\n3. **Check that the local server is running**\n   - You can test it with:\n     ```bash\n     curl http://localhost:11434\n     ```\n   - You should get a response showing available models.\n\n4. **Use it from Python**\n   - Install the client:\n     ```bash\n     pip install ollama\n     ```\n   - Example:\n     ```python\n     import ollama\n\n     response = ollama.chat(\n         model=\'llama3\',\n         messages=[{"role": "user", "content": "Hello!"}]\n     )\n\n     print(response[\'mes

# Encouraging multiple searches

In [ ]:
# The model often answers after the first search, even when more searches would help. 
# It reasons that it already knows enough, so why bother. We push it to explore more by rewriting the instructions.

In [82]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
iteration #2...
function_call: search {"query":"certificate submit project while accepting submissions live cohort self-paced FAQ join course"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while submissions are still open. You can also work through the material in self-paced mode, but certificate eligibility depends on finishing with a live cohort and completing the project/peer review while that cohort is accepting them.

If you want, I can also help with:
- how to start from scratch,
- what you need for the certificate,
- or where to submit the project.


'Yes — you can still join the course.\n\nIf you want a certificate, the key thing is to submit your project while submissions are still open. You can also work through the material in self-paced mode, but certificate eligibility depends on finishing with a live cohort and completing the project/peer review while that cohort is accepting them.\n\nIf you want, I can also help with:\n- how to start from scratch,\n- what you need for the certificate,\n- or where to submit the project.'

# Restricting off-topic questions

In [83]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen gambit"}
iteration #2...
function_call: search {"query":"queen gambit chess opening definition"}
function_call: search {"query":"what is the Queen's Gambit chess"}
function_call: search {"query":"Queen's Gambit opening chess pawn move d4 d5 c4"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening**.

It starts with:
1. **d4 d5**
2. **c4**

White offers the **c-pawn** to try to control the center and tempt Black’s **d-pawn** away from the center. Despite the name, it’s usually **not a real sacrifice** because White often gets the pawn back or gains strong positional pressure.

There are two main forms:
- **Queen’s Gambit Accepted**: Black takes the c-pawn
- **Queen’s Gambit Declined**: Black does not take it

If you want, I can also explain:
- the **idea behind the opening**
- the **difference between Accepted and Declined**
- or a **simple move-by-move example**


'The **Queen’s Gambit** is a classic **chess opening**.\n\nIt starts with:\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** to try to control the center and tempt Black’s **d-pawn** away from the center. Despite the name, it’s usually **not a real sacrifice** because White often gets the pawn back or gains strong positional pressure.\n\nThere are two main forms:\n- **Queen’s Gambit Accepted**: Black takes the c-pawn\n- **Queen’s Gambit Declined**: Black does not take it\n\nIf you want, I can also explain:\n- the **idea behind the opening**\n- the **difference between Accepted and Declined**\n- or a **simple move-by-move example**'

In [84]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry relevant to “queen gambit,” so I can’t answer that from the course materials.

If you meant a course topic, could you rephrase it or use the exact lecture/assignment name? Are there other areas you want to explore?


'I couldn’t find a course FAQ entry relevant to “queen gambit,” so I can’t answer that from the course materials.\n\nIf you meant a course topic, could you rephrase it or use the exact lecture/assignment name? Are there other areas you want to explore?'

In [ ]:
# This handwritten loop is the best way to understand what frameworks hide from you. 
# Every agent framework wraps this same pattern, whether it's LangChain, PydanticAI, or the OpenAI Agents SDK.

In [ ]:
# ToyAIKit is a teaching and experimentation library, and it is NOT meant for production use. 

In [85]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [86]:
# We register our search function along with the schema from earlier lessons:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [87]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [88]:
search

<function __main__.search(query)>

In [89]:
# Writing that schema by hand is annoying, and we don't want to do it for every function. So we don't have to.
# If we add a type hint and a docstring to search, ToyAIKit reads them and derives the schema for us:

In [90]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [91]:
# we just pass the search fucntion
agent_tools = Tools()
agent_tools.add_tool(search)

In [92]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [93]:
# The output is the same JSON schema we hand-wrote in the function calling lesson. 
# ToyAIKit generated it from the docstring and the type hint.
# Every modern agent framework does this same trick. It reads a typed Python function with a docstring and builds the schema from it. 
# The OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this way.
# You write the tool and the framework figures out how to describe it.

In [95]:
# Create the chat interface and a callback, then build the runner:

In [97]:
chat_interface = IPythonChatInterface() # The chat_interface handles display in the notebook. 
callback = DisplayingRunnerCallback(chat_interface) # The callback renders model messages and tool calls as they happen.
 
# The runner runs the agent loop, the same while True we wrote by hand. 
# It sends messages, executes function calls, adds tool outputs back, and repeats until the model is done.
# We pick gpt-5.4-mini here on purpose. 
# Without it, ToyAIKit falls back to a smaller, faster default that doesn't follow the instructions as reliably.

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [98]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [99]:
result.cost

CostInfo(input_cost=Decimal('0.00324825'), output_cost=Decimal('0.0014445'), total_cost=Decimal('0.00469275'))

In [100]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"run Olama locally"}', call_id='call_RuH4BY4xETBSbYLXpI5gULgh',